# 编排器-工人（Orchestrator-Workers）工作流

## 简介

你是否遇到过这种情况：同一任务需要多个视角，却无法提前预判哪些视角最有价值？编排器-工人模式解决了这个问题：由一个中心 LLM 分析每个具体任务，动态决定把哪些子任务委派给专门化的工人 LLM。

传统做法要么需要人工多次编写提示词，要么使用硬编码的并行化，无论上下文如何都生成相同的变体。

在这种方法中，编排器 LLM 先分析任务，判断哪些变体对当前具体情况最有价值，然后再委派给工人 LLM 分别生成每个变体。

### 你将构建什么

一个接收产品描述请求的系统，它会：

1. 分析哪几类营销文案有价值
2. 为工人生成专门化的任务描述
3. 产出针对受众优化的多种内容变体
4. 汇总并返回所有工人的协调结果

### 前置条件

- Python 3.9 或更高版本
- 将 Anthropic API 密钥设置为环境变量：`export ANTHROPIC_API_KEY='your-key'`
- 对提示词工程有基本理解
- 熟悉 Python 类和类型提示


### 何时使用此工作流

此工作流适合无法提前预判所需子任务的复杂任务。它与简单并行化的关键区别在于灵活性——子任务不是预先定义的，而是由编排器根据具体输入决定的。

**适合使用此模式的情形：**

- 任务需要多种不同的方法或视角
- 最优子任务取决于具体输入
- 你需要比较不同的策略或风格

**不适合使用此模式的情形：**

- 简单的单输出任务（会带来不必要的复杂性）
- 对时延要求高（多次 LLM 调用会增加开销）
- 子任务可预测且可以预先定义（此时应使用更简单的并行化）

## 工作原理

编排器-工人模式分为两个阶段运行：

1. **分析与规划阶段**：编排器 LLM 接收任务和上下文，分析哪些方法有价值，并生成 XML 格式的结构化子任务描述。

2. **执行阶段**：每个工人 LLM 会收到：
   - 用于提供上下文的原始任务
   - 分配给它的具体子任务类型和描述
   - 提供的任何其他上下文

编排器*在运行时*决定创建哪些子任务，这使它比预定义的并行工作流更具自适应性。

## 设置

### 安装
```bash
pip install anthropic
```

### 辅助函数
本示例使用 `util.py` 中的辅助函数来发起 LLM 调用并解析 XML 响应：

- `llm_call(prompt, system_prompt="", model="claude-sonnet-4-6")`：向 Claude 发送提示词并返回文本响应
- `extract_xml(text, tag)`：使用正则表达式从 XML 标签中提取内容

这些工具函数负责处理 API 认证（从环境变量读取 `ANTHROPIC_API_KEY`），并为编排器-工人模式提供简洁的接口。完整实现见 [util.py](util.py)。

## 实现

`FlexibleOrchestrator` 类协调整个两阶段工作流：

**关键设计决策：**
- 提示词是接受运行时变量（`task`、`context`）的模板，以保证灵活性
- 使用 XML 进行结构化输出解析（该格式可靠且对语言模型友好）
- 工人同时接收原始任务和它自己的具体指令，以获得更好的上下文
- 错误处理会校验工人是否返回了非空响应

实现中包含：
- `parse_tasks()`：把编排器的 XML 输出解析为结构化的任务字典
- `FlexibleOrchestrator.process()`：主协调逻辑，先调用编排器，再调用工人
- 响应校验，用于捕获并处理工人的空输出

In [ ]:
from util import extract_xml, llm_call

# Model configuration
MODEL = "claude-sonnet-4-6"  # Fast, capable model for both orchestrator and workers


def parse_tasks(tasks_xml: str) -> list[dict]:
    """Parse XML tasks into a list of task dictionaries."""
    tasks = []
    current_task = {}

    for line in tasks_xml.split("\n"):
        line = line.strip()
        if not line:
            continue

        if line.startswith("<task>"):
            current_task = {}
        elif line.startswith("<type>"):
            current_task["type"] = line[6:-7].strip()
        elif line.startswith("<description>"):
            current_task["description"] = line[12:-13].strip()
        elif line.startswith("</task>"):
            if "description" in current_task:
                if "type" not in current_task:
                    current_task["type"] = "default"
                tasks.append(current_task)

    return tasks


class FlexibleOrchestrator:
    """Break down tasks and run them in parallel using worker LLMs."""

    def __init__(
        self,
        orchestrator_prompt: str,
        worker_prompt: str,
        model: str = MODEL,
    ):
        """Initialize with prompt templates and model selection."""
        self.orchestrator_prompt = orchestrator_prompt
        self.worker_prompt = worker_prompt
        self.model = model

    def _format_prompt(self, template: str, **kwargs) -> str:
        """Format a prompt template with variables."""
        try:
            return template.format(**kwargs)
        except KeyError as e:
            raise ValueError(f"Missing required prompt variable: {e}") from e

    def process(self, task: str, context: dict | None = None) -> dict:
        """Process task by breaking it down and running subtasks in parallel."""
        context = context or {}

        # Step 1: Get orchestrator response
        orchestrator_input = self._format_prompt(self.orchestrator_prompt, task=task, **context)
        orchestrator_response = llm_call(orchestrator_input, model=self.model)

        # Parse orchestrator response
        analysis = extract_xml(orchestrator_response, "analysis")
        tasks_xml = extract_xml(orchestrator_response, "tasks")
        tasks = parse_tasks(tasks_xml)

        print("\n" + "=" * 80)
        print("ORCHESTRATOR ANALYSIS")
        print("=" * 80)
        print(f"\n{analysis}\n")

        print("\n" + "=" * 80)
        print(f"IDENTIFIED {len(tasks)} APPROACHES")
        print("=" * 80)
        for i, task_info in enumerate(tasks, 1):
            print(f"\n{i}. {task_info['type'].upper()}")
            print(f"   {task_info['description']}")

        print("\n" + "=" * 80)
        print("GENERATING CONTENT")
        print("=" * 80 + "\n")

        # Step 2: Process each task
        worker_results = []
        for i, task_info in enumerate(tasks, 1):
            print(f"[{i}/{len(tasks)}] Processing: {task_info['type']}...")

            worker_input = self._format_prompt(
                self.worker_prompt,
                original_task=task,
                task_type=task_info["type"],
                task_description=task_info["description"],
                **context,
            )

            worker_response = llm_call(worker_input, model=self.model)
            worker_content = extract_xml(worker_response, "response")

            # Validate worker response - handle empty outputs
            if not worker_content or not worker_content.strip():
                print(f"⚠️  Warning: Worker '{task_info['type']}' returned no content")
                worker_content = f"[Error: Worker '{task_info['type']}' failed to generate content]"

            worker_results.append(
                {
                    "type": task_info["type"],
                    "description": task_info["description"],
                    "result": worker_content,
                }
            )

        # Display results
        print("\n" + "=" * 80)
        print("RESULTS")
        print("=" * 80)
        for i, result in enumerate(worker_results, 1):
            print(f"\n{'-' * 80}")
            print(f"Approach {i}: {result['type'].upper()}")
            print(f"{'-' * 80}")
            print(f"\n{result['result']}\n")

        return {
            "analysis": analysis,
            "worker_results": worker_results,
        }

## 示例用例：生成营销文案变体

接下来通过一个实际示例来看编排器-工人模式的运行：为某个产品生成多种风格的营销文案。

**为什么这个例子能很好地体现该模式：**
- 不同产品适合不同的营销角度
- 「最佳」变体取决于具体的产品特性和目标受众
- 编排器能根据输入调整策略，而不是套用固定模板

**提示词设计说明：**
- 编排器提示词要求给出 2-3 种方法，并提供 XML 结构指引
- 工人提示词给工人完整上下文（原始任务、它的风格以及编写准则）
- 两个提示词都使用清晰的 XML 格式，以确保可靠解析

In [9]:
ORCHESTRATOR_PROMPT = """
Analyze this task and break it down into 2-3 distinct approaches:

Task: {task}

Return your response in this format:

<analysis>
Explain your understanding of the task and which variations would be valuable.
Focus on how each approach serves different aspects of the task.
</analysis>

<tasks>
    <task>
    <type>formal</type>
    <description>Write a precise, technical version that emphasizes specifications</description>
    </task>
    <task>
    <type>conversational</type>
    <description>Write an engaging, friendly version that connects with readers</description>
    </task>
</tasks>
"""

WORKER_PROMPT = """
Generate content based on:
Task: {original_task}
Style: {task_type}
Guidelines: {task_description}

Return your response in this format:

<response>
Your content here, maintaining the specified style and fully addressing requirements.
</response>
"""


orchestrator = FlexibleOrchestrator(
    orchestrator_prompt=ORCHESTRATOR_PROMPT,
    worker_prompt=WORKER_PROMPT,
)

results = orchestrator.process(
    task="Write a product description for a new eco-friendly water bottle",
    context={
        "target_audience": "environmentally conscious millennials",
        "key_features": ["plastic-free", "insulated", "lifetime warranty"],
    },
)


ORCHESTRATOR ANALYSIS


This task requires creating marketing copy for an eco-friendly water bottle. The core challenge is balancing product information with persuasive messaging while highlighting the environmental benefits. Different approaches would serve distinct marketing channels and audience segments:

1. A **feature-focused technical approach** would appeal to detail-oriented consumers who make purchasing decisions based on specifications, materials, and measurable environmental impact. This serves e-commerce listings and comparison shopping.

2. A **lifestyle-oriented emotional approach** would connect with values-driven consumers through storytelling and aspirational messaging, emphasizing how the product fits into an eco-conscious lifestyle. This serves social media and brand-building content.

3. A **benefit-driven practical approach** would focus on solving everyday problems while weaving in sustainability advantages, appealing to mainstream consumers who want both functi

## 总结

现在你已经实现了一个能根据具体输入动态调整任务拆分的编排器-工人模式。该模式生成了多种营销文案变体——每种都针对不同的受众和场景——而无需你预先定义这些变体应该是什么。

### 关键要点

**模式收益：**
- **适应性**：编排器为每个具体输入决定最佳方法
- **灵活性**：只需更换提示词，就能轻松应用到不同领域
- **结构化协调**：基于 XML 的通信确保了可靠解析
- **错误恢复能力**：校验能捕获并处理工人的失败

**此模式擅长的场景：**
- 需要多种视角的内容生成（营销、文档、创意写作）
- 能从不同分析视角中获益的分析任务
- 拆分策略取决于问题本身的问题求解

### 局限性与注意事项

**成本与时延：**
- 需要 N+1 次 LLM 调用（1 次编排器 + N 次工人）
- 当前实现是顺序处理（工人逐个运行）
- 若要提升性能，可考虑用 `asyncio` 或线程池并行化工人调用

**不适合使用此模式的情形：**
- 输出单一且明确的简单任务（增加的复杂性不值得）
- 对时延敏感的应用（多次 API 调用会增加开销）
- 子任务始终相同的任务（应改用预定义的并行化）

**需要考虑的故障模式：**
- 编排器可能拆得不理想（提示词工程至关重要）
- 工人可能返回空或格式错误的响应（我们用校验来处理这种情况）
- 如果模型没有严格遵循格式，XML 解析可能失败（可考虑改用 JSON 作为替代）

### 后续步骤

**增强此实现：**
1. 使用 `asyncio` 并行执行工人调用，以提升性能
2. 为失败的工人实现重试逻辑
3. 增加一个综合阶段，由 LLM 合并各工人的输出
4. 尝试不同的编排器策略（例如要求更多或更少的子任务）

**适配到你的用例：**
- 修改编排器提示词，为你的领域引导任务拆分
- 调整工人提示词，提供领域特定的指令
- 添加与你的应用相关的上下文参数
- 考虑用 Claude Opus 担任编排器、Claude Haiku 担任工人，以平衡成本与质量